# Trust-region B: five-iteration diagnostic run

This notebook runs the isolated `stackelberg.trust_region_b` variant for five trust-region iterations. It loads the final 120-dimensional NN price vector from `results/000`, enables one COPT network-state initialization before each local Adam solve, and writes all new outputs below `results/000/trust_region_b_five_iterations`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pandapower.networks as pn

from stackelberg.followers.bg import create_cigre_residential_network
from stackelberg.followers.ip import create_industrial_park_network
from stackelberg.network import create_case33bw_20kv
from stackelberg.trust_region_b import (
    TrustRegionExperimentConfig,
    run_trust_region_experiment,
)

In [2]:
T = 24
SCENARIOS = ["SB1", "SB2", "BG1", "BG2", "IP1"]
MG_BUSES = [17, 32, 15, 21, 1]
RESULTS_DIR = Path("results/000")
OUTPUT_DIR = RESULTS_DIR / "trust_region_b_five_iterations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

history_candidates = sorted(RESULTS_DIR.glob("NN_solver_history_*.csv"))
if not history_candidates:
    raise FileNotFoundError(f"No NN history found below {RESULTS_DIR}.")
warm_start_path = max(history_candidates, key=lambda path: path.stat().st_mtime_ns)
nn_history = pd.read_csv(warm_start_path)
if nn_history.empty:
    raise ValueError(f"NN history is empty: {warm_start_path}")
pi_columns = sorted(
    (column for column in nn_history if column.startswith("pi_")),
    key=lambda column: int(column.removeprefix("pi_")),
)
if len(pi_columns) != len(SCENARIOS) * T:
    raise ValueError(
        f"Expected {len(SCENARIOS) * T} prices, got {len(pi_columns)}."
    )
last_nn_row = nn_history.sort_values("iteration").iloc[-1]
warm_start = last_nn_row[pi_columns].to_numpy(dtype=float)
nn_true_objective = float(last_nn_row["true_loss"])

print("Warm start:", warm_start_path)
print("Saved NN iteration:", int(last_nn_row["iteration"]))
print("Warm-start true objective:", nn_true_objective)
display(pd.DataFrame(warm_start.reshape(len(SCENARIOS), T), index=SCENARIOS))

Warm start: results/000/NN_solver_history_20260912-234954.csv
Saved NN iteration: 9800
Warm-start true objective: -103.73745796679384


,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
SB1,0.905496,0.972395,0.992483,0.998625,0.982330,0.981594,1.006895,1.002889,0.991757,1.000325,...,1.037887,1.018056,1.024587,0.997190,0.993872,1.024512,1.423613,1.006008,1.086199,0.996892
SB2,0.984292,0.985921,0.971553,0.977133,0.962063,0.958349,1.004334,0.994941,1.498616,0.988460,...,1.011563,0.999690,0.996986,0.981880,1.001129,1.013935,0.991371,0.986417,1.089251,0.993309
BG1,1.499927,1.499906,1.499724,1.499684,1.499929,1.499869,1.499946,1.499939,1.499915,1.499948,...,1.499942,1.499932,1.499939,1.499932,1.499938,1.499940,1.499914,0.261827,1.499944,1.499898
BG2,1.499915,1.499862,1.499867,1.499870,1.499940,1.499835,1.499953,1.499933,1.499934,1.499927,...,1.499954,1.499947,1.499947,1.499948,1.499952,1.499953,1.499886,1.499871,1.499922,1.499468
IP1,1.493962,1.488898,1.495653,1.494458,1.490675,1.476853,1.496844,1.499135,1.499072,1.499319,...,1.499453,1.499741,1.499321,1.497359,1.494321,1.498004,1.496904,1.494693,1.498408,1.498219


In [3]:
config = TrustRegionExperimentConfig(
    upper_net=create_case33bw_20kv(),
    lower_nets=[
        pn.create_cigre_network_lv(),
        pn.create_cigre_network_lv(),
        create_cigre_residential_network(),
        create_cigre_residential_network(),
        create_industrial_park_network(),
    ],
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=42,
    device="cpu",
    x0=warm_start,
    lower_time_limit=30.0,
    lower_mip_gap=1e-3,
    copt_initialize_network_state=True,
    tr_params={
        "R_0": 0.05,
        "R_max": 1.0,
        "gamma_dec": 0.7,
        "gamma_inc": 1.5,
        "eta_1": 0.1,
        "eta_2": 0.75,
    },
    algo_params={
        "n_samples": 60,
        "K_max": 5,
        "sampling_method": "lhs",
        "residual_update_mode": "accumulate",
        "rbf_geometry_condition_max": 1e12,
        "random_seed": 42,
        "solve_params": {
            "steps_init": 0,
            "steps_warm": 1000,
            "lr": 1e-2,
            "penalty_weights": {
                "eq": 1000.0,
                "ineq": 100.0,
                "soc": 100.0,
                "bound": 10.0,
            },
            "tr_weight": 1e6,
            "verbose": False,
            "diagnostic_trace_every": 25,
        },
    },
    residual_model_params={
        "model_type": "rbf",
        "rbf_kernel": "cubic",
        "rbf_smoothing": 1e-3,
        "rbf_degree": 1,
        "rbf_scale": "trust_radius",
    },
    save_results=True,
    output_dir=str(OUTPUT_DIR),
    save_true_responses=False,
)

print("K_max:", config.algo_params["K_max"])
print("steps_warm:", config.algo_params["solve_params"]["steps_warm"])
print("COPT initialization:", config.copt_initialize_network_state)

K_max: 5
steps_warm: 1000
COPT initialization: True


In [4]:
tr_result = run_trust_region_experiment(config)
print("Completed iterations:", tr_result["n_iter"])
print("Elapsed seconds:", tr_result["elapsed_seconds"])
print("Saved paths:", tr_result.get("saved_paths"))

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Using global surrogate to solve initial point...
Initial surrogate solve used steps_init=0. Time taken to solve initial point 0.0655 seconds.
Iteration 0 | R=0.0500 
Building 60-point local designs for each of 5 followers (center + coordinate star + L

In [5]:
history = pd.DataFrame(tr_result["history"])
history["Pred"] = history["F_tilde_current"] - history["F_tilde_candidate"]
history["Ared"] = history["F_current"] - history["F_candidate"]
history["candidate_distance_inf"] = [
    float(np.max(np.abs(np.asarray(candidate) - np.asarray(center))))
    for candidate, center in zip(history["x_cand"], history["x_k"])
]

diagnostics_history = tr_result.get("optimizer_diagnostics_history") or []
(OUTPUT_DIR / "optimizer_diagnostics_history.json").write_text(
    json.dumps(diagnostics_history, indent=2), encoding="utf-8"
)

trace_frames = []
diagnostic_rows = []
for tr_iteration, diagnostics in enumerate(diagnostics_history):
    trace_frame = pd.DataFrame(diagnostics.get("trace", []))
    if not trace_frame.empty:
        trace_frame.insert(0, "tr_iteration", tr_iteration)
        trace_frames.append(trace_frame)
    initial = diagnostics.get("initial") or {}
    final = diagnostics.get("final") or {}
    best = diagnostics.get("best") or {}
    diagnostic_rows.append({
        "tr_iteration": tr_iteration,
        "copt_warm_start_seconds": diagnostics.get("copt_warm_start_seconds"),
        "copt_warm_start_objective": diagnostics.get("copt_warm_start_objective"),
        "initial_merit": initial.get("merit"),
        "initial_weighted_penalty": initial.get("weighted_penalty"),
        "initial_gradient_l2": initial.get("gradient_l2"),
        "best_merit": best.get("merit"),
        "best_raw_surrogate_objective": best.get("raw_surrogate_objective"),
        "final_merit": final.get("merit"),
        "final_raw_surrogate_objective": final.get("raw_surrogate_objective"),
        "final_weighted_penalty": final.get("weighted_penalty"),
    })

optimizer_trace = (
    pd.concat(trace_frames, ignore_index=True)
    if trace_frames
    else pd.DataFrame()
)
if not optimizer_trace.empty:
    optimizer_trace.to_csv(
        OUTPUT_DIR / "optimizer_trace_all_iterations.csv", index=False
    )
diagnostic_frame = pd.DataFrame(diagnostic_rows)
diagnostic_frame.to_csv(
    OUTPUT_DIR / "optimizer_diagnostics_by_iteration.csv", index=False
)

best_true_objective = float(
    np.min(
        np.concatenate([
            history["F_current"].to_numpy(dtype=float),
            history["F_candidate"].to_numpy(dtype=float),
        ])
    )
)
summary = {
    "warm_start_path": str(warm_start_path.resolve()),
    "warm_start_true_objective": nn_true_objective,
    "completed_iterations": int(tr_result["n_iter"]),
    "first_F_current": float(history.iloc[0]["F_current"]),
    "final_F_next": float(history.iloc[-1]["F_next"]),
    "best_observed_true_objective": best_true_objective,
    "decisions": history["decision"].tolist(),
    "final_radius": float(history.iloc[-1]["R"]),
    "copt_warm_start_seconds": [
        item.get("copt_warm_start_seconds")
        for item in diagnostics_history
    ],
    "copt_warm_start_objectives": [
        item.get("copt_warm_start_objective")
        for item in diagnostics_history
    ],
    "elapsed_seconds": float(tr_result["elapsed_seconds"]),
}
(OUTPUT_DIR / "five_iteration_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

display(pd.DataFrame([summary]))
display(history[[
    "k", "F_current", "F_candidate", "F_tilde_current",
    "F_tilde_candidate", "Pred", "Ared", "rho", "decision",
    "R_before", "R", "candidate_distance_inf",
    "surrogate_error_current", "surrogate_error_candidate",
]])
display(diagnostic_frame)


,warm_start_path,warm_start_true_objective,completed_iterations,first_F_current,final_F_next,best_observed_true_objective,decisions,final_radius,copt_warm_start_seconds,copt_warm_start_objectives,elapsed_seconds
0,/Users/liqi/Desktop/Codes/Surrogated-assisted ...,-103.737458,5,-103.827705,-104.855473,-104.855473,"[accept_shrink, reject, accept_keep, accept_sh...",0.01715,"[0.038371916161850095, 0.03857299988158047, 0....","[-103.67925309114439, -104.14892818313383, -10...",2374.796


,k,F_current,F_candidate,F_tilde_current,F_tilde_candidate,Pred,Ared,rho,decision,R_before,R,candidate_distance_inf,surrogate_error_current,surrogate_error_candidate
0,0,-103.827705,-104.133523,-103.679253,-110.779502,7.100249,0.305818,0.043071,accept_shrink,0.05000,0.03500,0.05000,0.148452,6.645979
1,1,-104.133523,-103.877820,-104.148928,-105.702710,1.553781,-0.255703,-0.164568,reject,0.03500,0.02450,0.03500,0.015405,1.824889
2,2,-104.133523,-104.465598,-104.156240,-105.574594,1.418354,0.332075,0.234127,accept_keep,0.02450,0.02450,0.02450,0.022717,1.108996
3,3,-104.465853,-104.548795,-104.433600,-105.293700,0.860100,0.082942,0.096434,accept_shrink,0.02450,0.01715,0.02450,0.032253,0.744904
4,4,-104.548795,-104.855473,-104.526769,-105.283303,0.756534,0.306678,0.405372,accept_keep,0.01715,0.01715,0.01715,0.022026,0.427830


,tr_iteration,copt_warm_start_seconds,copt_warm_start_objective,initial_merit,initial_weighted_penalty,initial_gradient_l2,best_merit,best_raw_surrogate_objective,final_merit,final_raw_surrogate_objective,final_weighted_penalty
0,0,0.038372,-103.679253,-103.679268,1.481948e-09,24.873911,-117.301697,-124.025963,-116.357590,-124.060638,7.703054
1,1,0.038573,-104.148928,-104.148422,1.999712e-06,25.294842,-111.983063,-118.947372,-111.581223,-119.062279,7.481054
2,2,0.032155,-104.156240,-104.155716,1.998860e-06,25.250034,-112.137932,-118.863029,-111.486763,-118.880753,7.393982
3,3,0.032201,-104.433600,-104.433907,1.969366e-06,24.880909,-112.040756,-118.516312,-111.284637,-118.773056,7.488420
4,4,0.032380,-104.526769,-104.526100,6.697820e-06,26.609922,-112.036514,-118.518890,-111.209206,-118.830688,7.621489
